# SmolVLM2-2.2B LoRA Fine-tuning

Supervised LoRA fine-tuning of **SmolVLM2-2.2B-Instruct** on BangladeshiVQA, reproducing the
configuration reported in the paper's **Appendix B**. The full hyperparameters are also
recorded in [`finetuning_config.md`](finetuning_config.md), and the resulting adapter ships in
[`smolvlm2_lora_adapter/`](smolvlm2_lora_adapter/).

Training uses the **official 5,628-pair train split** (`annotations/splits/train.json`), so no
validation or test image is ever seen during training. The preview sample bundled in this
repository is **not** enough to train — download the full dataset (images + annotations) from
Hugging Face first, then point the paths below at it (or set `BVQA_IMAGE_DIR` / `BVQA_TRAIN_JSON`).

The paper trained on **dual NVIDIA T4 GPUs**: run `torchrun --nproc_per_node=2 train.py` to
reproduce that, or `python train.py` for a single GPU. Set `HF_TOKEN` (and `HF_REPO`) in your
environment only if you want to push the trained adapter to the Hub.

In [ ]:
!pip install -q -U "transformers>=4.50.0" accelerate peft bitsandbytes pillow

## Configuration & data check

Reads the same paths and defaults as `train.py`, then confirms the train split and its images
are visible before launching the run. Override the defaults with the `BVQA_IMAGE_DIR` and
`BVQA_TRAIN_JSON` environment variables.

In [ ]:
import os
import json

IMAGE_DIR = os.environ.get("BVQA_IMAGE_DIR", "images")
TRAIN_JSON = os.environ.get("BVQA_TRAIN_JSON", "annotations/splits/train.json")

train_data = json.load(open(TRAIN_JSON, encoding="utf-8"))
image_ids = sorted({row["image_id"] for row in train_data})
missing = [i for i in image_ids if not os.path.exists(os.path.join(IMAGE_DIR, f"{i}.jpg"))]

print("Train QA pairs:", len(train_data))
print("Unique training images:", len(image_ids))
print("Missing image files:", len(missing))

## Training script

Written to `train.py` so it can be launched as a clean process (single GPU with `python`, or
dual-GPU with `torchrun --nproc_per_node=2`). It loads the 4-bit NF4 base model, attaches a LoRA
adapter (rank 8, alpha 16, dropout 0.05 on the q/k/v/o projections), and trains for 3 epochs with
the 8-bit paged AdamW optimizer at learning rate 2e-4, logging every 5 steps.

In [ ]:
%%writefile train.py
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import (AutoProcessor, AutoModelForImageTextToText,
                          BitsAndBytesConfig, TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
IMAGE_DIR = os.environ.get("BVQA_IMAGE_DIR", "images")
TRAIN_JSON = os.environ.get("BVQA_TRAIN_JSON", "annotations/splits/train.json")
OUTPUT_DIR = os.environ.get("BVQA_OUTPUT_DIR", "smolvlm2-bangla-lora")
HF_REPO = os.environ.get("HF_REPO", "your-username/SmolVLM2-BangladeshiVQA-LoRA")
HF_SUBFOLDER = "finetune-lora"

with open(TRAIN_JSON, encoding="utf-8") as f:
    train_data = json.load(f)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.bfloat16)

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.do_image_splitting = False

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=bnb, _attn_implementation="sdpa",
    device_map={"": int(os.environ.get("LOCAL_RANK", 0))})

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none", task_type="CAUSAL_LM"))
model.config.use_cache = False

image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")


class VQADataset(Dataset):
    def __init__(self, rows):
        self.rows = [r for r in rows
                     if os.path.exists(os.path.join(IMAGE_DIR, f'{r["image_id"]}.jpg'))]
        self.cache = {}
        print("Verified training samples:", len(self.rows))

    def __len__(self):
        return len(self.rows)

    def _image(self, image_id):
        if image_id not in self.cache:
            image = Image.open(os.path.join(IMAGE_DIR, f"{image_id}.jpg")).convert("RGB")
            image.thumbnail((384, 384), Image.Resampling.LANCZOS)
            self.cache[image_id] = image
        return self.cache[image_id]

    def __getitem__(self, i):
        row = self.rows[i]
        return {"messages": [
            {"role": "user", "content": [
                {"type": "image", "image": self._image(row["image_id"])},
                {"type": "text", "text": f'{row["question"]}\nশুধুমাত্র উত্তর দিন।'}]},
            {"role": "assistant", "content": [
                {"type": "text", "text": row["answer"]}]}]}


def collate_fn(batch):
    texts, images = [], []
    for example in batch:
        messages = example["messages"]
        texts.append(processor.apply_chat_template(messages, add_generation_prompt=False).strip())
        images.append([next(c["image"] for c in messages[0]["content"] if c["type"] == "image")])
    out = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = out["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    out["labels"] = labels
    return out


args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    fp16=False,
    gradient_checkpointing=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=VQADataset(train_data),
                  data_collator=collate_fn)
trainer.train()

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Saved adapter to", OUTPUT_DIR)

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"])
    model.push_to_hub(HF_REPO, subfolder=HF_SUBFOLDER, token=os.environ["HF_TOKEN"], private=True)
    processor.push_to_hub(HF_REPO, subfolder=HF_SUBFOLDER, token=os.environ["HF_TOKEN"])
    print("Pushed adapter to", f"{HF_REPO}/{HF_SUBFOLDER}")

In [ ]:
!python train.py